# ryGPT — Kaggle T4×2 training (7B-Instruct)

Fine-tune **Qwen2.5-7B-Instruct** with QLoRA on your anonymized WhatsApp data.

## Why the 7B *Instruct* model (vs the earlier base-1.5B run)

See `.docs/DECISIONS.md` ADR-011. Short version:
- The base 1.5B model never learned that chat turns *end* — generations ran forever and drifted into foreign-script gibberish. The **Instruct** model is chat-tuned: `<|im_end|>` is already a registered stop token AND it was trained to emit it, so it **stops on its own**. The whole "never stops" problem is fixed at the source.
- 7B is much more coherent than 1.5B — the actual fix for "replies don't make sense."

## Before you run

In the right sidebar, confirm:
1. **Accelerator: GPU T4 ×2** (both GPUs — `accelerate launch --multi_gpu` uses them in parallel)
2. **Internet: On** (needed to download the ~15GB model)
3. **Add Data** → your private `rygpt-data` dataset (`train.jsonl`, `val.jsonl`, `name_mapping.json`)
4. **(Continuing a multi-session run only)** **Add Data** → **Your Notebooks** → this notebook's previous saved output. §6b auto-detects and resumes.

## Multi-session training

7B is ~4–5× slower per step than 1.5B, so this spans **several 12h sessions**. `scripts/train_7b.py` checkpoints every 1000 steps and auto-resumes from the latest checkpoint in `--out-dir`. To carry a checkpoint forward:
1. Before the session ends: **Save Version → Quick Save**, and set **Save output → "Save output for this version"** so `models/lora_adapter_7b/checkpoint-*` is preserved.
2. Next session: attach that output (step 4 above), run all cells top-to-bottom. §6b restores the checkpoint; training continues where it stopped.

## 1. Environment check

In [ ]:
!nvidia-smi | head -20
import torch
print()
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print('GPU count:', n)
    for i in range(n):
        cap = torch.cuda.get_device_capability(i)
        print(f'  [{i}] {torch.cuda.get_device_name(i)}  compute {cap}  '
              f'{"bf16" if cap[0] >= 8 else "fp16"}')
    if n < 2:
        print('WARNING: only 1 GPU visible — set Accelerator to GPU T4 x2 for 2x speed.')
else:
    raise SystemExit('No GPU detected. Settings -> Accelerator -> GPU T4 x2')

## 2. Verify internet is on

Needed to download the ~15GB base model.

In [ ]:
import urllib.request
try:
    urllib.request.urlopen('https://github.com', timeout=5)
    print('Internet: OK')
except Exception as e:
    raise SystemExit('Internet appears OFF. Settings -> Internet -> On, then re-run.\n' + str(e))

## 3. Clone the repo

Idempotent — safe to re-run. Steps out of the repo dir before deleting it so a stale cwd can't wedge the shell.

In [ ]:
import os, shutil
os.chdir('/kaggle/working')  # never sit inside the dir we're about to delete
REPO_DIR = '/kaggle/working/ryGPT'
if os.path.exists(REPO_DIR):
    print(f'{REPO_DIR} exists — removing and re-cloning for a clean state')
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 https://github.com/rihaans/ryGPT.git {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline
!ls

## 4. Install dependencies

Pinned `transformers<5` + `peft==0.17.*` (peft 0.18+ breaks checkpoint resume). The cell hard-verifies the pins so a bad environment fails HERE, not hours into training.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes

import transformers, peft
import bitsandbytes as bnb
print('transformers:', transformers.__version__)
print('peft:        ', peft.__version__)
print('bitsandbytes:', bnb.__version__)
assert transformers.__version__.split('.')[0] == '4', (
    f'transformers {transformers.__version__} but <5 required — pip downgrade did not take. '
    'Restart the session and re-run from the top.')
assert peft.__version__.startswith('0.17.'), (
    f'peft {peft.__version__} but 0.17.* required — 0.18+ breaks checkpoint resume.')
print('Environment OK — safe to train and resume.')

## 5. Wire the Kaggle dataset into the expected paths

Auto-detects `/kaggle/input/**/train.jsonl` regardless of dataset slug.

In [ ]:
import glob, os, shutil

REPO_DIR = '/kaggle/working/ryGPT'
os.chdir(REPO_DIR)

candidates = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
if not candidates:
    print('Could not find train.jsonl under /kaggle/input/.')
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            print(root)
            for f in files[:10]:
                print('   ', f)
    raise SystemExit('Attach the rygpt-data dataset (Add Data -> rygpt-data), then re-run.')

dataset_dir = os.path.dirname(candidates[0])
print('Found dataset at:', dataset_dir)

processed_dir = os.path.join(REPO_DIR, 'data', 'processed')
anon_dir = os.path.join(REPO_DIR, 'data', 'anonymized')
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(anon_dir, exist_ok=True)

for name, dst in [
    ('train.jsonl', os.path.join(processed_dir, 'train.jsonl')),
    ('val.jsonl', os.path.join(processed_dir, 'val.jsonl')),
    ('name_mapping.json', os.path.join(anon_dir, 'name_mapping.json')),
]:
    m = glob.glob(f'{dataset_dir}/**/{name}', recursive=True)
    if not m:
        print(f'  MISSING: {name}  (name_mapping.json is optional)')
        continue
    shutil.copy(m[0], dst)
    print(f'  {name}  ({os.path.getsize(m[0])/1e6:.1f} MB)  ->  {dst}')

!ls -lh {processed_dir}

## 6. Sanity-check the data

Confirms the trainer can read the JSONL before we spend hours on it.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ryGPT')
from src.dataset import read_jsonl, example_to_chat_messages

train = read_jsonl('data/processed/train.jsonl')
val = read_jsonl('data/processed/val.jsonl')
print(f'train: {len(train):,}  |  val: {len(val):,}')
print()
for m in example_to_chat_messages(train[0]):
    print(f'  [{m["role"]:>9}] {m["content"][:100]}')

## 6b. Resume from a previous session (optional)

If you attached a previous session's output (step 4), this finds the latest `checkpoint-*` under `/kaggle/input/` and copies it into `models/lora_adapter_7b/` so training resumes. On a fresh run it prints a message and does nothing — safe to always run.

In [ ]:
import glob, os, shutil

REPO_DIR = '/kaggle/working/ryGPT'
os.chdir(REPO_DIR)

# Previous session's output contains models/lora_adapter_7b/checkpoint-<step>/.
checkpoint_dirs = glob.glob('/kaggle/input/**/lora_adapter_7b/checkpoint-*', recursive=True)
if checkpoint_dirs:
    step = lambda p: int(os.path.basename(p.rstrip('/')).split('checkpoint-')[-1])
    checkpoint_dirs.sort(key=step)
    latest = checkpoint_dirs[-1]
    src_dir = os.path.dirname(latest)                      # the lora_adapter_7b/ dir
    dst_dir = os.path.join(REPO_DIR, 'models', 'lora_adapter_7b')
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    print(f'Restored checkpoint from previous session: {src_dir}')
    print(f'  latest step: {step(latest)}')
    print('Training will resume from this checkpoint automatically.')
else:
    print('No previous checkpoint found under /kaggle/input/ — starting fresh.')

## 7. Train (Phase 6, 7B)

Launched via `accelerate launch --multi_gpu --num_processes 2` so both T4s train in parallel.

7B-on-T4 hyperparameters (in `scripts/train_7b.py`):
- `--batch-size 2 --grad-accum 8` per GPU → effective batch 2×8×2 = 32 (matches the 1.5B run). **If you hit CUDA OOM, change to `--batch-size 1 --grad-accum 16`.**
- `--epochs 1` (the 1.5B eval_loss bottomed near epoch 1.5, so >1 overfits; a 7B instruct model needs even less to adapt style).
- `--eval-steps 1000 --save-steps 1000` → checkpoints every 1000 steps (caps lost work if a session dies).

**This will not finish in one 12h session.** When time runs low: Save Version (with output) → Stop → resume next session (§6b handles it).

In [ ]:
!accelerate launch --multi_gpu --num_processes 2 scripts/train_7b.py \
    --base-model Qwen/Qwen2.5-7B-Instruct \
    --batch-size 2 \
    --grad-accum 8 \
    --max-seq-length 256 \
    --epochs 1 \
    --eval-steps 1000 \
    --save-steps 1000 \
    --logging-steps 50 \
    --wandb-disabled \
    --out-dir /kaggle/working/ryGPT/models/lora_adapter_7b

## 8. Evaluate (Phase 7)

`--load-4bit` (two 7B fp16 copies would need ~30GB — 4-bit fits a T4) and `--skip-base` (skip the untuned comparison — saves a second model copy and a full perplexity pass; we mainly care whether the tuned model is good). Produces `eval/perplexity.md`, `eval/samples.md`, `eval/memorization.md`.

Run this only once training has fully finished (or from a fresh session with the final adapter attached + restored via §6b).

In [ ]:
!python scripts/07_evaluate.py \
    --adapter-dir models/lora_adapter_7b \
    --load-4bit \
    --skip-base

## 9. Preview eval results inline

In [ ]:
from pathlib import Path
for name in ('perplexity', 'memorization', 'samples'):
    p = Path(f'/kaggle/working/ryGPT/eval/{name}.md')
    if p.exists():
        print('=' * 20, f'eval/{name}.md', '=' * 20)
        text = p.read_text(encoding='utf-8')
        print(text[:4000])
        if len(text) > 4000:
            print(f'\n... ({len(text) - 4000} more chars in the file)')
        print()

## 10. Package adapter for download

Bundles adapter + tokenizer + eval into one archive under `/kaggle/working/` (shown as Output in the right sidebar).

In [ ]:
!cd /kaggle/working/ryGPT && tar czf /kaggle/working/rygpt_lora_adapter_7b.tar.gz models/lora_adapter_7b eval
!ls -lh /kaggle/working/rygpt_lora_adapter_7b.tar.gz

## 11. Download to your laptop

1. Right sidebar → **Output** → download `rygpt_lora_adapter_7b.tar.gz`.
2. On your laptop:

```powershell
cd C:\Users\Rihaan\Development\Personal\ryGPT\ryGPT
tar -xzf rygpt_lora_adapter_7b.tar.gz
python scripts/chat.py --adapter-dir models/lora_adapter_7b
```

7B inference needs a GPU (or lots of patience on CPU). Your 4070 laptop runs it fine; the office PC (no GPU) will be slow — use it only for a quick check. `chat.py` already auto-detects GPU vs CPU.

## Stop the session

Not finished? **Save Version** (with output) *before* Stop, so the next session resumes (§6b). Then **Stop** to release the T4×2 back to your weekly quota (T4×2 burns quota ~2× as fast per hour).